# AICE Associate 변주 문제 v3 - 경매 낙찰가, 예상 판매 가격 예측 (회귀)

### 이 노트북의 구성요소

아래 다이어그램은 이 노트북 해설에서 실제로 사용된 함수·클래스·모듈을 구간별로 모은 것입니다 (굵게 표시된 이름은 이 노트북에서 특별히 선택된 항목입니다).

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>

<div style="display:flex; flex-direction:column; gap:6px; width:fit-content;">
<pre class="mermaid">
flowchart LR
    subgraph LIB["라이브러리"]
        LIB1["<b>라이브러리</b><br/>numpy<br/>pandas<br/>pyplot<br/>seaborn"]
    end
    subgraph EXP["데이터 탐색"]
        EXP1["<b>데이터 탐색</b><br/>groupby<br/>head<br/>merge<br/>read_csv"]
    end
    subgraph PREP["전처리"]
        PREP1["<b>전처리</b><br/><b>MinMaxScaler</b><br/>LabelEncoder<br/>drop<br/>fillna<br/>get_dummies<br/>info"]
    end
    LIB --> EXP --> PREP
    classDef libCls fill:#F0EEE8,stroke:#8A8478,color:#3A362E
    class LIB libCls
    classDef expCls fill:#E9F1F6,stroke:#5B7C99,color:#1E3548
    class EXP expCls
    classDef prepCls fill:#FAF3E7,stroke:#B08D57,color:#5C4720
    class PREP prepCls
</pre>
<pre class="mermaid">
flowchart LR
    subgraph ML["머신러닝"]
        ML1["<b>머신러닝</b><br/><b>XGBRegressor</b><br/><b>r2_score</b><br/>DataFrame<br/>DecisionTreeRegressor<br/>GridSearchCV<br/>sort_values"]
    end
    subgraph UNSUP["비지도학습"]
        UNSUP1["<b>비지도학습</b><br/>(기본 구성)"]
    end
    subgraph DL["딥러닝"]
        DL1["<b>딥러닝</b><br/><b>ModelCheckpoint</b><br/><b>load_model</b><br/>Dense<br/>Dropout<br/>EarlyStopping<br/>Sequential"]
    end
    ML --> UNSUP --> DL
    classDef mlCls fill:#FBEEEA,stroke:#B5654A,color:#5C2A1B
    class ML mlCls
    classDef unsupCls fill:#F3EEF4,stroke:#8B6F8E,color:#402F42
    class UNSUP unsupCls
    classDef dlCls fill:#EBEEF6,stroke:#445577,color:#232C3D
    class DL dlCls
</pre>
</div>

<script>
mermaid.initialize({ startOnLoad: false, securityLevel: "loose" });
mermaid.run();
</script>
"""))

### 스톱워치

아래 셀을 실행하면 경과 시간이 표시됩니다. 문제를 풀기 시작하기 전에 먼저 실행하세요.

In [ ]:
from IPython.display import HTML, display

def show_stopwatch():
    display(HTML('''
<div id="stopwatch" style="font-size: 18px; font-weight: bold; color: #d93025; padding: 8px; border: 1px solid #d93025; border-radius: 5px; display: inline-block;">
    ⏱️ 경과 시간: <span id="time_str">00:00</span>
</div>
<script>
    var sec = 0;
    var timer = setInterval(function(){
        sec++;
        var m = Math.floor(sec / 60);
        var s = sec % 60;
        document.getElementById("time_str").innerText =
            (m < 10 ? "0" + m : m) + ":" + (s < 10 ? "0" + s : s);
    }, 1000);
</script>
'''))
show_stopwatch()

### 시나리오

대한민국 주요 농산물 생산자 협동조합은 다가오는 추수 시즌에 발생할 수 있는 시장 변동성을 예측하여 최적의 경매 가격을 설정하고자 합니다. 이 모델을 통해 각 품목별 예상 낙찰가를 미리 파악함으로써, 생산자는 안정적인 수익을 확보하고 유통 과정에서 불필요한 손실을 최소화할 수 있습니다. 또한, 이를 바탕으로 미래 계약에 대한 리스크 관리 전략을 수립합니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 문제를 다 풀면 `## 자동 채점` 셀을 실행해서 점수를 확인할 수 있습니다 (해설을 보기 전에 먼저 채점해보세요).
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.
- 데이터 로더: `read_csv` / 기본모델: `DecisionTreeRegressor` / 비교모델: `XGBRegressor` / 스케일러: `MinMaxScaler`

**[데이터 컬럼 설명]**

- auction_price : 경매 낙찰가, 예상 판매 가격
- total_yield : 총 수확량 (톤 단위)
- crop_type : 주요 작물 종류
- region_code : 생산 지역 코드 (5~8개 코드값) (dim.csv 와 병합 키)
- harvest_date : 피처 컬럼
- quality_score : 피처 컬럼
- storage_cost : 피처 컬럼
- weather_index : 피처 컬럼
- record_id : 식별자(모델링에 불필요)
- (병합 후) dim_value : 지역별 평균 경매 가중치

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요 (문제 데이터 2개 테이블을 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_tables(seed=4019, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    main_df = pd.DataFrame({'total_yield': base.round(2)})

    cats = [f"Cat{i+1}" for i in range(rng.integers(3, 5))]
    main_df['crop_type'] = rng.choice(cats, size=n)

    n_regions = rng.integers(5, 9)
    regions = [f"R{i+1:02d}" for i in range(n_regions)]
    main_df['region_code'] = rng.choice(regions, size=n)

    for col in ['harvest_date', 'quality_score', 'storage_cost', 'weather_index']:
        main_df[col] = rng.normal(0, 1, size=n).round(2)

    main_df['record_id'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    classes = ['경매 낙찰가', '예상 판매 가격']
    if '회귀' == "분류":
        if False and len(classes) >= 3:
            bins = np.quantile(z, [1 / 3, 2 / 3])
            idx = np.digitize(z, bins)
            main_df['auction_price'] = [classes[i] for i in idx]
        else:
            prob = 1 / (1 + np.exp(-z))
            labels = (rng.random(n) < prob).astype(int)
            main_df['auction_price'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        main_df['auction_price'] = (base * 1.5 + noise).round(2)

    for col in ['total_yield'] + ['harvest_date', 'quality_score', 'storage_cost', 'weather_index'][:1]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        main_df.loc[na_idx, col] = np.nan

    main_df = main_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    dim_df = pd.DataFrame({
        'region_code': regions,
        "dim_value": rng.uniform(0.5, 2.0, size=n_regions).round(3),
    })
    return main_df, dim_df


main_df, dim_df = synthesize_tables()
main_df.to_csv("data.csv", index=False)
dim_df.to_csv("dim.csv", index=False)
print("데이터 저장 완료 - data.csv:", main_df.shape, "/ dim.csv:", dim_df.shape)
main_df.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

pandas, numpy, matplotlib.pyplot, seaborn 을 각각 pd, np, plt, sns 별칭으로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드 (read_csv/read_json)

`data.csv` 를 `pd.read_csv` 로 읽어 **my_data** 에, `dim.csv` 를 `pd.read_csv` 로 읽어 **dim_data** 에 각각 할당하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 결측치 확인

my_data 의 `total_yield` 컬럼에 결측치(NaN)가 몇 개 있는지 `isna().sum()` 으로 확인하세요. 몇 개입니까?

In [ ]:
# (3) 정답을 answer_3 변수에 저장하세요 (실행하세요)

answer_3 = ""


### 4. 데이터 병합 (pd.merge)

`region_code` 를 키로 my_data 와 dim_data 를 **left join** 하여 **data_merged** 에 저장하세요.

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



### 5. 데이터 집계 (groupby)

`crop_type` 별 `total_yield` 의 평균을 구해 **df_grp** 에 저장하세요.

In [ ]:
# (5) 여기에 답안코드를 작성하고 실행하세요



### 6. 시각화 (subplots)

주요 작물 종류(crop_type) 분포의 countplot 과, 경매 낙찰가, 예상 판매 가격 별 총 수확량 (톤 단위)(total_yield) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='crop_type', ax=axes[0])
sns.histplot(data=data_merged, x='total_yield', hue='auction_price', ax=axes[1])
plt.show()
```

In [ ]:
# (6) 정답을 answer_6 변수에 저장하세요 (실행하세요)

answer_6 = ""


### 7. 시각화 2

경매 낙찰가, 예상 판매 가격 별 총 수확량 (톤 단위)(total_yield) 분포를 seaborn boxplot 으로 시각화하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 8. 이상치 처리

IQR 기준(K=2.0)을 벗어나는 이상치 행을 제거하고, 식별자 컬럼도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 data_temp 를 사용합니다).

```python
q1 = data_merged['total_yield'].quantile(0.25)
q3 = data_merged['total_yield'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.(A)(data_merged[(data_merged['total_yield'] > upper_fence) | (data_merged['total_yield'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['record_id'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (8) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 9. 결측치 처리

다음은 data_temp 의 결측치를 대표값으로 채우는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fill_value = data_temp['total_yield'].mode()[0]
data_na = data_temp.fillna({'total_yield': fill_value})
if data_na['harvest_date'].isnull().any():
    data_na['harvest_date'] = data_na['harvest_date'].fillna(data_na['harvest_date'].mode()[0])
```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 10. 인코딩

`crop_type` 는 원-핫 인코딩(get_dummies, drop_first=True), `region_code` 는 sklearn LabelEncoder 로 인코딩해서 data_preset 에 저장하세요.

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. 데이터 분리

auction_price 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.25, random_state=7
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (11) 여기에 답안코드를 작성하고 실행하세요



### 12. 스케일링

MinMaxScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train) 
# 오류 삽입 지점: X_train_scaled에 불필요한 정수 나눗셈을 시도하여 값을 왜곡시킴
X_train_scaled = X_train_scaled / 2
X_valid_scaled = scaler.transform(X_valid)
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 스케일러 특성

MinMaxScaler 를 훈련 데이터에 적용하면, 결과 값의 분포는 이론적으로 어떤 특성을 가지게 됩니까?

In [ ]:
# (13) 정답을 answer_13 변수에 저장하세요 (실행하세요)

answer_13 = ""


## <AI 모델링>

### 14. 머신러닝 기본 (fit-predict)

DecisionTreeRegressor 로 모델을 하나 만들어 학습시키고, 검증데이터에 대한 예측값을 **pred_y** 에 저장하세요. (변수명: model, pred_y)

In [ ]:
# (14) 여기에 답안코드를 작성하고 실행하세요



### 15. GridSearch 모델링

DecisionTreeRegressor 와 XGBRegressor 를 GridSearchCV(cv=5)로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- XGBRegressor 의 n_estimators 후보: [50, 100]
- 변수명: gs_a (베이스 모델), gs_b (비교 모델)

In [ ]:
# (15) 여기에 답안코드를 작성하고 실행하세요



### 16. GridSearch 결과 확인

위 GridSearch에서 XGBRegressor 의 n_estimators 후보는 [50, 100] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (16) 정답을 answer_16 변수에 저장하세요 (실행하세요)

answer_16 = ""


### 17. 변수중요도

XGBRegressor 의 변수중요도 Top 15개(정렬 ascending=False)를 뽑아 bar 차트로 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': X_train.columns.values}) 
fi = fi.sort_values('importance', ascending=False)[:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()
```

In [ ]:
# (17) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 18. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **r2_score** 를 각각 계산해서 a_score, b_score 에 저장하세요.

In [ ]:
# (18) 여기에 답안코드를 작성하고 실행하세요



### 19. 모델 성능 비교

바로 위에서 계산한 a_score 와 b_score 를 비교했을 때, 어느 모델이 더 우수하다고 판단할 수 있습니까? (둘 중 하나를 실행 결과에 따라 답하세요)

In [ ]:
# (19) 정답을 answer_19 변수에 저장하세요 (실행하세요)

answer_19 = ""


### 20. 딥러닝 설계

다음은 딥러닝 모델을 설계하는 코드입니다. EarlyStopping 에서 '몇 epoch 동안 개선이 없으면 멈출지' 지정하는 파라미터 이름(빈칸 **(A)**)을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 model/cb_list 를 사용합니다). (은닉층 활성함수: relu, 출력층: linear/mse, ModelCheckpoint 포함)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
cb_list = [EarlyStopping(monitor='val_loss', (A)=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))
```

In [ ]:
# (20) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 21. 딥러닝 학습

위에서 설계한 model 을 batch_size=32, epochs=50 으로 학습하고 history 에 저장하세요 (callbacks=cb_list 사용).

In [ ]:
# (21) 여기에 답안코드를 작성하고 실행하세요



### 22. 학습곡선 시각화

history 를 이용해서 학습/검증 **mae** 변화를 한 그래프에 시각화하세요 (x축 라벨: epoch, 범례 위치: upper left, 범례 텍스트: train/val).

In [ ]:
# (22) 여기에 답안코드를 작성하고 실행하세요



### 23. 저장된 모델 재사용

ModelCheckpoint 로 저장된 `best_model.keras` 를 `load_model()` 로 다시 불러와서, 검증데이터에 대한 예측값을 **reload_pred** 에 저장하세요.

In [ ]:
# (23) 여기에 답안코드를 작성하고 실행하세요



---
## 자동 채점

위 문항을 모두 풀고 실행한 뒤, 아래 셀을 실행해서 점수를 확인하세요. (딥러닝 문항은 tensorflow 가 필요하므로 Colab 에서 실행했을 때만 채점됩니다.)

In [ ]:
_CHECKS = [(1, '라이브러리 임포트', "all(k in globals() for k in ['pd','np','plt','sns'])", '임포트 확인됨', 'pd/np/plt/sns 임포트가 필요합니다'), (2, '데이터 로드 (read_csv/read_json)', "'my_data' in globals() and tuple(getattr(my_data,'shape',())) == (600, 9) and 'dim_data' in globals() and tuple(getattr(dim_data,'shape',())) == (8, 2)", 'shape 일치', 'my_data shape (600, 9), dim_data shape (8, 2) 이어야 합니다'), (3, '결측치 확인', "_norm_answer(globals().get('answer_3')) == '18'", '정답 일치', '정답은 18 입니다'), (4, '데이터 병합 (pd.merge)', "'data_merged' in globals() and tuple(getattr(data_merged,'shape',())) == (600, 10)", 'shape 일치', 'data_merged shape (600, 10) 이어야 합니다'), (5, '데이터 집계 (groupby)', "'df_grp' in globals() and sorted(str(x) for x in getattr(df_grp,'index',[])) == ['Cat1', 'Cat2', 'Cat3', 'Cat4']", '그룹 일치', 'df_grp 의 그룹(index)이 올바르지 않습니다'), (6, '시각화 (subplots)', "_norm_answer(globals().get('answer_6')) == 'subplots'", '정답 일치', '정답은 subplots 입니다'), (8, '이상치 처리', "'data_temp' in globals() and tuple(getattr(data_temp,'shape',())) == (588, 9)", 'shape 일치', 'data_temp shape (588, 9) 이어야 합니다 ((A)=drop, 전체 코드를 실행했는지 확인)'), (9, '결측치 처리', "'data_na' in globals() and int(data_na.isna().sum().sum()) == 0 and tuple(getattr(data_na,'shape',())) == (588, 9)", '결측치 제거 및 shape 일치', '결측치가 남아있거나 shape 이 올바르지 않습니다'), (10, '인코딩', "'data_preset' in globals() and data_preset.select_dtypes(include='object').shape[1] == 0 and tuple(getattr(data_preset,'shape',())) == (588, 11)", '인코딩 및 shape 일치', '문자열(object) 컬럼이 남아있거나 shape 이 올바르지 않습니다'), (11, '데이터 분리', "all(k in globals() for k in ['X_train','X_valid','y_train','y_valid']) and len(X_train) == 441 and len(X_valid) == 147", '분리 크기 일치', 'X_train/X_valid 길이가 441/147 이어야 합니다'), (12, '스케일링', "'X_train_scaled' in globals() and 'X_valid_scaled' in globals() and len(X_train_scaled) == len(X_train) and len(X_valid_scaled) == len(X_valid)", '스케일링 변수 확인', 'X_train_scaled/X_valid_scaled 를 올바르게 생성했는지 확인하세요'), (13, '스케일러 특성', "len(_norm_answer(globals().get('answer_13'))) > 2", '답변 작성됨 (해설과 비교해 직접 확인하세요)', 'answer_13 가 비어있습니다'), (14, '머신러닝 기본 (fit-predict)', "'model' in globals() and hasattr(model,'predict') and 'pred_y' in globals() and len(pred_y) == len(X_valid)", 'model/pred_y 확인', 'model, pred_y 를 올바르게 생성했는지 확인하세요'), (15, 'GridSearch 모델링', "'gs_a' in globals() and hasattr(gs_a,'best_estimator_') and 'gs_b' in globals() and hasattr(gs_b,'best_estimator_')", 'gs_a/gs_b 학습 확인', 'gs_a, gs_b 가 GridSearchCV로 학습되었는지 확인하세요'), (16, 'GridSearch 결과 확인', "_norm_answer(globals().get('answer_16')) == '100'", '정답 일치', '정답은 100 입니다'), (17, '변수중요도', "'fi' in globals() and set(['feature','importance']).issubset(set(fi.columns)) and len(fi) == 10", 'fi 구조 확인', 'fi 에 feature/importance 컬럼과 올바른 길이가 필요합니다'), (18, '성능평가', "'a_score' in globals() and 'b_score' in globals() and isinstance(a_score,(int,float)) and isinstance(b_score,(int,float))", 'a_score/b_score 확인', 'a_score, b_score 를 숫자로 계산했는지 확인하세요'), (19, '모델 성능 비교', "len(_norm_answer(globals().get('answer_19'))) > 0", '답변 작성됨 (해설과 비교해 직접 확인하세요)', 'answer_19 가 비어있습니다'), (20, '딥러닝 설계', "_norm_answer(globals().get('answer_20')) == 'patience' and 'model' in globals() and hasattr(model,'fit') and 'cb_list' in globals()", '정답 및 model/cb_list 확인', "answer_20 는 'patience' 여야 하고, model/cb_list 도 함께 생성해야 합니다 (Colab 실행 필요)"), (21, '딥러닝 학습', "'history' in globals() and hasattr(history,'history')", 'history 확인 (Colab 실행 필요)', 'history 를 생성했는지 확인하세요 (Colab 실행 필요)'), (23, '저장된 모델 재사용', "'reload_pred' in globals() and 'X_valid_scaled' in globals() and len(reload_pred) == len(X_valid_scaled)", 'reload_pred 확인 (Colab 실행 필요)', 'reload_pred 를 생성했는지 확인하세요 (Colab 실행 필요)')]

def _norm_answer(x):
    if x is None:
        return ""
    try:
        return str(x).strip().strip('"\'').lower()
    except Exception:
        return ""

_score = 0
_total = 0
_report = []
for _n, _title, _cond, _ok_msg, _fail_msg in _CHECKS:
    _total += 1
    try:
        _result = bool(eval(_cond))
    except Exception:
        _result = False
    if _result:
        _score += 1
        _report.append(f"[PASS] {_n}번 {_title}: {_ok_msg}")
    else:
        _report.append(f"[FAIL] {_n}번 {_title}: {_fail_msg}")

print("\n".join(_report))
print(f"\n총점: {_score} / {_total}  ({_score/_total*100:.0f}점)" if _total else "채점 가능한 문항이 없습니다")


---
## 해설

채점 후 아래에서 정답과 설명을 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

> import ... as ... 문법으로 널리 쓰이는 관례적 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 (read_csv/read_json) [코드작성]

In [ ]:
my_data = pd.read_csv('data.csv')
dim_data = pd.read_csv('dim.csv')
my_data.head(4)

> pd.read_csv() 로 파일 형식에 맞는 로더를 사용합니다.

### 3번 해설 - 결측치 확인 [결과값예측]

In [ ]:
18

> Series.isna().sum() 은 True(결측치)의 개수를 셉니다.

### 4번 해설 - 데이터 병합 (pd.merge) [코드작성]

In [ ]:
data_merged = pd.merge(my_data, dim_data, on='region_code', how='left')
data_merged.head(4)

> pd.merge(left, right, on=키, how='left') 는 왼쪽 테이블 기준으로 오른쪽 테이블을 결합합니다.

### 5번 해설 - 데이터 집계 (groupby) [코드작성]

In [ ]:
df_grp = data_merged.groupby('crop_type')['total_yield'].mean()
df_grp

> groupby(기준컬럼)[대상컬럼].mean() 형태로 그룹별 집계를 구합니다.

### 6번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='crop_type', ax=axes[0])
sns.histplot(data=data_merged, x='total_yield', hue='auction_price', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 7번 해설 - 시각화 2 [코드작성]

In [ ]:
sns.boxplot(data=data_merged, x='auction_price', y='total_yield')
plt.show()

> sns.boxplot(x=, y=) / sns.jointplot(x=, y=) 형태로 그립니다.

### 8번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = data_merged['total_yield'].quantile(0.25)
q3 = data_merged['total_yield'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.drop(data_merged[(data_merged['total_yield'] > upper_fence) | (data_merged['total_yield'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['record_id'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 9번 해설 - 결측치 처리 [오류정정]

In [ ]:
fill_value = data_temp['total_yield'].mode()[0]
data_na = data_temp.fillna({'total_yield': fill_value})
data_na['harvest_date'] = data_na['harvest_date'].fillna(data_na['harvest_date'].mode()[0])

> [loop_control] 원래 코드는 모든 결측치를 연속적으로 채우는 단순한 순차 처리였으나, 버그 코드에서는 'harvest_date'에 결측치가 존재하는지 확인하는 조건문(`if data_na['harvest_date'].isnull().any():`)을 추가했습니다. 이 조건이 만족되지 않으면 두 번째(실제 데이터 채우기) 로직이 완전히 건너뛰어지게 되어, 특정 데이터셋에서는 'harvest_date'가 제대로 채워지지 않을 수 있습니다.

### 10번 해설 - 인코딩 [코드작성]

In [ ]:
data_preset = pd.get_dummies(data=data_na, columns=['crop_type'], drop_first=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset['region_code'] = le.fit_transform(data_preset['region_code'])
data_preset.info()

> 저-카디널리티는 원-핫, 코드성 범주는 라벨 인코딩을 흔히 사용합니다.

### 11번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop(['auction_price'], axis=1)
y = data_preset['auction_price']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=7)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 12번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [operator_precedence] X_train_scaled 변수에 대해 fit_transform 결과를 받은 후, 의도치 않게 해당 값들을 2로 나누는 연산을 추가했습니다. 이는 스케일링된 데이터의 실제 값을 왜곡하여 모델 학습에 잘못된 입력이 들어가게 만듭니다.

### 13번 해설 - 스케일러 특성 [결과값예측]

In [ ]:
'최솟값 0, 최댓값 1 사이 구간으로 정규화됩니다.'

> MinMaxScaler 의 정의에 따른 이론적 특성입니다.

### 14번 해설 - 머신러닝 기본 (fit-predict) [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=7)
model.fit(X_train_scaled, y_train)
pred_y = model.predict(X_valid_scaled)
pred_y[:5]

> import → model = 클래스() → model.fit(X_train, y_train) → pred_y = model.predict(X_valid) 4줄 템플릿입니다.

### 15번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

gs_a = GridSearchCV(DecisionTreeRegressor(random_state=7), {'max_depth':[3,5,7]}, cv=5)
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(XGBRegressor(random_state=7), {'n_estimators':[50, 100], 'max_depth':[3,5,7]}, cv=5)
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다. XGBRegressor 는 sklearn 기본 앙상블 외에 XGBoost/LightGBM 계열일 수도 있습니다.

### 16번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
100

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 17번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()

> [scope_reference] 원래 코드는 모델의 실제 중요도(`feature_importances_`)를 사용했으나, 오류 버전에서는 중요도 열에 대신 학습 데이터의 컬럼 값(`X_train.columns.values`)을 잘못 할당했습니다. 이는 피처 이름과 중요도 값이 서로 연결되지 않아 시각화 결과가 의미 없는 값을 나타내게 됩니다.

### 18번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import r2_score

y_pred_a = gs_a.best_estimator_.predict(X_valid_scaled)
y_pred_b = gs_b.best_estimator_.predict(X_valid_scaled)

a_score = r2_score(y_valid, y_pred_a)
b_score = r2_score(y_valid, y_pred_b)
print(a_score, b_score)

> sklearn.metrics.r2_score 에 (실제값, 예측값) 순서로 인자를 넣습니다.

### 19번 해설 - 모델 성능 비교 [결과값예측]

In [ ]:
'실행 결과에 따라 달라집니다: 값이 더 큰 쪽 이 더 우수한 모델입니다. a_score, b_score 를 직접 비교해서 판단하세요.'

> 정확도/F1/ROC-AUC 등은 높을수록, MAE/MSE 등 오차 지표는 낮을수록 좋은 성능입니다.

### 20번 해설 - 딥러닝 설계 [빈칸채우기]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
cb_list = [EarlyStopping(monitor='val_loss', patience=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))

> EarlyStopping(patience=N) 은 N번 연속 개선이 없으면 학습을 멈춥니다. 정답: patience

### 21번 해설 - 딥러닝 학습 [코드작성]

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32,
                    validation_data=(X_valid_scaled, y_valid), callbacks=cb_list)

> model.fit(X, y, epochs=, batch_size=, validation_data=, callbacks=) 형태입니다.

### 22번 해설 - 학습곡선 시각화 [코드작성]

In [ ]:
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('Model mae')
plt.xlabel('epoch')
plt.ylabel('mae')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

> history.history[지표] 로 epoch별 기록을 꺼내 plt.plot() + plt.legend(loc=...) 으로 그립니다.

### 23번 해설 - 저장된 모델 재사용 [코드작성]

In [ ]:
from tensorflow.keras.models import load_model

saved_model = load_model('best_model.keras')
reload_pred = saved_model.predict(X_valid_scaled)
reload_pred[:5]

> 학습 없이도 저장된 가중치를 불러와(load_model) 바로 predict() 할 수 있습니다.